In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import datetime
import glob
import hashlib
import os
from urllib.request import urlretrieve

In [2]:
def parse_csv_env(name, default):
    raw_value = os.environ.get(name, default)
    return [item.strip() for item in raw_value.split(',') if item.strip()]

services = parse_csv_env('SERVICES', 'yellow,green')
start_year = int(os.environ.get('START_YEAR', '2015'))
end_year = int(os.environ.get('END_YEAR', '2025'))
months = [int(item) for item in parse_csv_env('MONTHS', '1,2,3,4,5,6,7,8,9,10,11,12')]
run_id = os.environ.get('RUN_ID', datetime.datetime.now().strftime("%Y%m%d_%H%M%S"))

In [3]:
def log_step(message):
    timestamp = datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')
    print(f'[{timestamp}Z] {message}')


def build_spark(app_name):
    jars_dir = os.environ.get('SPARK_JARS_DIR', '/home/jovyan/jars')
    jar_paths = sorted(glob.glob(os.path.join(jars_dir, '*.jar')))
    builder = (
        SparkSession.builder
        .appName(app_name)
        .config('spark.sql.session.timeZone', 'UTC')
    )
    if jar_paths:
        jars_csv = ','.join(jar_paths)
        classpath = ':'.join(jar_paths)
        log_step(f'Loading JARs: {jar_paths}')
        builder = (
            builder
            .config('spark.jars', jars_csv)
            .config('spark.driver.extraClassPath', classpath)
            .config('spark.executor.extraClassPath', classpath)
        )
    else:
        log_step(f'No JARs found in {jars_dir}')
    return builder.getOrCreate()


def assert_snowflake_connector(spark):
    class_names = [
        'net.snowflake.spark.snowflake.DefaultSource',
        'snowflake.DefaultSource',
    ]
    for class_name in class_names:
        try:
            spark.sparkContext._jvm.java.lang.Class.forName(class_name)
            log_step(f'Snowflake connector detected: {class_name}')
            return
        except Exception:
            continue
    raise RuntimeError('Snowflake connector not found. Run `bash scripts/download_snowflake_jars.sh` from the repo root, restart the container, and rerun this notebook.')


app = build_spark('01_ingesta_parquet_raw')
assert_snowflake_connector(app)

# Credenciales Snowflake desde variables de ambiente
sf_option = {
    'sfURL': os.environ['SF_HOST'],
    'sfUser': os.environ['SF_USER'],
    'sfPassword': os.environ['SF_PASSWORD'],
    'sfDatabase': os.environ['SF_DATABASE'],
    'sfSchema': os.environ.get('SF_RAW_SCHEMA', 'raw'),
    'sfWarehouse': os.environ.get('SF_WAREHOUSE', ''),
    'sfRole': os.environ.get('SF_ROLE', ''),
}

log_step(f'Run configured: services={services}, years={start_year}-{end_year}, months={months}, run_id={run_id}')

[2026-04-05 21:56:48Z] Loading JARs: ['/home/jovyan/jars/snowflake-jdbc-3.28.0.jar', '/home/jovyan/jars/spark-snowflake_2.12-3.1.8.jar']
[2026-04-05 21:56:50Z] Snowflake connector detected: net.snowflake.spark.snowflake.DefaultSource
[2026-04-05 21:56:50Z] Run configured: services=['yellow', 'green'], years=2015-2015, months=[2], run_id=20260405_test_02


In [4]:
def get_nyc_url(service, year, month):
    base_url = os.environ.get(f'{service.upper()}_BASE_URL', 'https://d37ci6vzurychx.cloudfront.net/trip-data').rstrip('/')
    return f"{base_url}/{service}_tripdata_{year}-{month:02d}.parquet"

In [5]:
def download_parquet(url, service, year, month):
    local_dir = os.environ.get('LOCAL_PARQUET_DIR', '/tmp/nyc_taxi_tripdata')
    os.makedirs(local_dir, exist_ok=True)
    local_path = os.path.join(local_dir, f'{service}_{year}_{month:02d}.parquet')
    if not os.path.exists(local_path):
        log_step(f'Downloading {url}')
        urlretrieve(url, local_path)
    else:
        log_step(f'Using cached parquet {local_path}')
    return local_path

In [6]:
def standardize_columns(df, target_columns):
    for field in df.schema.fields:
        if isinstance(field.dataType, TimestampNTZType):
            df = df.withColumn(field.name, F.col(field.name).cast(TimestampType()))

    for col_name, col_type in target_columns.items():
        if col_name in df.columns:
            df = df.withColumn(col_name, F.col(col_name).cast(col_type))
        else:
            df = df.withColumn(col_name, F.lit(None).cast(col_type))
    return df

In [7]:
def compute_trip_nk(df, service):
    columns = set(df.columns)

    def existing_col(*names):
        for name in names:
            if name in columns:
                return F.col(name)
        return F.lit(None)

    pickup_col = existing_col('tpep_pickup_datetime', 'lpep_pickup_datetime', 'pickup_datetime')
    dropoff_col = existing_col('tpep_dropoff_datetime', 'lpep_dropoff_datetime', 'dropoff_datetime')
    vendor_col = existing_col('VendorID', 'vendorid')
    pu_col = existing_col('PULocationID', 'pu_location_id')
    do_col = existing_col('DOLocationID', 'do_location_id')
    total_amount_col = existing_col('total_amount')

    return df.withColumn(
        'trip_nk',
        F.sha2(
            F.concat_ws(
                '||',
                F.lit(service),
                vendor_col.cast('string'),
                pickup_col.cast('string'),
                dropoff_col.cast('string'),
                pu_col.cast('string'),
                do_col.cast('string'),
                total_amount_col.cast('string')
            ),
            256
        )
    )

In [8]:
def log_audit(service, year, month, status, rows_read=0, rows_written=0, notes=""):
    audit_row = [(run_id, service, year, month, status, rows_read, rows_written, datetime.datetime.utcnow(), notes)]
    schema = "run_id string, service string, year int, month int, status string, rows_read long, rows_written long, event_timestamp timestamp, notes string"
    audit_df = app.createDataFrame(audit_row, schema)
    try:
        audit_df.write.format('snowflake').options(**sf_option).option('dbtable', 'RAW.LOAD_AUDIT').mode('append').save()
    except Exception as audit_error:
        log_step(f'Audit logging failed for service={service} year={year} month={month:02d}: {str(audit_error)[:200]}')

In [9]:
expected_schema = {
    'VendorID': IntegerType(),
    'tpep_pickup_datetime': TimestampType(),
    'tpep_dropoff_datetime': TimestampType(),
    'passenger_count': DoubleType(),
    'trip_distance': DoubleType(),
    'RatecodeID': DoubleType(),
    'PULocationID': IntegerType(),
    'DOLocationID': IntegerType(),
    'payment_type': LongType(),
    'fare_amount': DoubleType(),
    'total_amount': DoubleType()
}

In [10]:
for service in services:
    for year in range(start_year, end_year + 1):
        for month in months:
            url = get_nyc_url(service, year, month)

            try: 
                log_step(f'Starting batch service={service} year={year} month={month:02d}')
                local_path = download_parquet(url, service, year, month)
                df_raw = app.read.parquet(local_path)
                rows_read = df_raw.count()
                log_step(f'Rows read from parquet: {rows_read}')

                df_processed = standardize_columns(df_raw, expected_schema)

                df_processed = df_processed.withColumns({
                    "run_id": F.lit(run_id),
                    "service_type": F.lit(service),
                    "source_year": F.lit(year),
                    "source_month": F.lit(month),
                    "ingested_at_utc": F.current_timestamp(),
                    "source_path": F.lit(url)
                })

                df_processed = compute_trip_nk(df_processed, service)

                target_table = f'{service.upper()}_TRIPS'
                raw_schema = os.environ.get('SF_RAW_SCHEMA', 'RAW')
                preactions = f'DELETE FROM {raw_schema}.{target_table} WHERE source_year = {year} AND source_month = {month}'

                log_step(f'Writing batch to {raw_schema}.{target_table}')
                df_processed.write.format('snowflake').options(**sf_option).option('dbtable', target_table).option('preactions', preactions).mode('append').save()

                log_audit(service, year, month, "SUCCESS", rows_read, rows_read)
                log_step(f'Batch loaded successfully: service={service} year={year} month={month:02d}')
            except Exception as e:
                status = 'Missing' if '403' in str(e) or '404' in str(e) else 'ERROR'
                log_step(f'Batch failed: service={service} year={year} month={month:02d} status={status}')
                log_step(str(e)[:300])
                log_audit(service, year, month, status, 0, 0, notes=str(e)[:500])


[2026-04-05 21:56:50Z] Starting batch service=yellow year=2015 month=02
[2026-04-05 21:56:50Z] Downloading https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2015-02.parquet
[2026-04-05 21:57:00Z] Rows read from parquet: 12442394
[2026-04-05 21:57:00Z] Writing batch to RAW.YELLOW_TRIPS
[2026-04-05 22:01:03Z] Batch loaded successfully: service=yellow year=2015 month=02
[2026-04-05 22:01:03Z] Starting batch service=green year=2015 month=02
[2026-04-05 22:01:03Z] Downloading https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2015-02.parquet
[2026-04-05 22:01:05Z] Rows read from parquet: 1574830
[2026-04-05 22:01:05Z] Writing batch to RAW.GREEN_TRIPS
[2026-04-05 22:01:45Z] Batch loaded successfully: service=green year=2015 month=02


In [11]:
df_audit_summary = app.read.format("snowflake").options(**sf_option) \
    .option("dbtable", "LOAD_AUDIT").load() \
    .filter(F.col("run_id") == run_id)

df_audit_summary.groupBy("service", "status").count().show()

+-------+-------+-----+
|service| status|count|
+-------+-------+-----+
|  green|SUCCESS|    1|
| yellow|SUCCESS|    1|
+-------+-------+-----+

